In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

np.random.seed(2)

In [4]:
train_data = pd.read_csv("train.csv")
test_data = pd.read_csv("test.csv")

def feature_engineering(df):
    data = df.copy()
    data['Title'] = data['Name'].apply(lambda x: x.split(',')[1].split('.')[0].strip())
    title_mapping = {'Mr':0, 'Miss':1, 'Mrs':2, 'Master':3, 'Dr':4, 'Rev':4, 
                     'Col':4, 'Major':4, 'Mlle':4, 'Countess':4, 'Ms':4, 
                     'Capt':4, 'Lady':4, 'Sir':4, 'Don':4, 'Jonkheer':4}
    data['Title'] = data['Title'].map(title_mapping).fillna(4).astype(int)
    data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
    data['IsAlone'] = (data['FamilySize'] == 1).astype(int)
    data['Age'] = data['Age'].fillna(data['Age'].median())
    data['Fare'] = data['Fare'].fillna(data['Fare'].median())
    data['Embarked'] = data['Embarked'].fillna(data['Embarked'].mode()[0])
    return data

train_processed = feature_engineering(train_data)
test_processed = feature_engineering(test_data)

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 
            'Embarked', 'Title', 'FamilySize', 'IsAlone']

train_encoded = pd.get_dummies(train_processed[features], 
                               columns=['Sex', 'Embarked'], drop_first=True)
test_encoded = pd.get_dummies(test_processed[features], 
                              columns=['Sex', 'Embarked'], drop_first=True)

train_encoded, test_encoded = train_encoded.align(test_encoded, join='left', axis=1, fill_value=0)

scaler = StandardScaler()
X_train = scaler.fit_transform(train_encoded.values.astype(float))
X_test = scaler.transform(test_encoded.values.astype(float))
y_train = train_data['Survived'].values.reshape(-1, 1)

In [5]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

def initialize_parameters(n_x, n_h, n_y):
    W1 = np.random.randn(n_h, n_x) * 0.01
    b1 = np.zeros((n_h, 1))
    W2 = np.random.randn(n_y, n_h) * 0.01
    b2 = np.zeros((n_y, 1))
    return {"W1": W1, "b1": b1, "W2": W2, "b2": b2}

def forward_propagation(X, parameters):
    W1, b1 = parameters["W1"], parameters["b1"]
    W2, b2 = parameters["W2"], parameters["b2"]
    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)
    Z2 = np.dot(W2, A1) + b2
    A2 = sigmoid(Z2)
    cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}
    return A2, cache

def backward_propagation(parameters, cache, X, Y, lambd=0.1):
    m = X.shape[1]
    W1, W2 = parameters["W1"], parameters["W2"]
    A1, A2 = cache["A1"], cache["A2"]
    Z1 = cache["Z1"]
    dZ2 = A2 - Y
    dW2 = (1/m) * np.dot(dZ2, A1.T) + (lambd/m) * W2
    db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)
    dZ1 = np.dot(W2.T, dZ2) * relu_derivative(Z1)
    dW1 = (1/m) * np.dot(dZ1, X.T) + (lambd/m) * W1
    db1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)
    grads = {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}
    return grads

def compute_cost(A2, Y, parameters, lambd=0.1):
    m = Y.shape[1]
    logprobs = Y * np.log(A2 + 1e-8) + (1 - Y) * np.log(1 - A2 + 1e-8)
    cross_entropy_cost = -np.sum(logprobs) / m
    W1 = parameters["W1"]
    W2 = parameters["W2"]
    l2_regularization = (lambd / (2 * m)) * (np.sum(np.square(W1)) + np.sum(np.square(W2)))
    return cross_entropy_cost + l2_regularization

def update_parameters(parameters, grads, learning_rate):
    parameters["W1"] = parameters["W1"] - learning_rate * grads["dW1"]
    parameters["b1"] = parameters["b1"] - learning_rate * grads["db1"]
    parameters["W2"] = parameters["W2"] - learning_rate * grads["dW2"]
    parameters["b2"] = parameters["b2"] - learning_rate * grads["db2"]
    return parameters

In [6]:
X = X_train.T
Y = y_train.T

n_x = X.shape[0]
n_h = 20
n_y = 1

parameters = initialize_parameters(n_x, n_h, n_y)
learning_rate = 0.5
lambd = 0.1
epochs = 5000

print(f"训练样本数: {X.shape[1]}，特征数: {n_x}")
print("-" * 50)

for i in range(epochs):
    A2, cache = forward_propagation(X, parameters)
    cost = compute_cost(A2, Y, parameters, lambd)
    grads = backward_propagation(parameters, cache, X, Y, lambd)
    parameters = update_parameters(parameters, grads, learning_rate)
    if i % 500 == 0:
        print(f"Epoch {i:4d}, Cost: {cost:.6f}")

print("-" * 50)
print("训练完成！")

训练样本数: 891，特征数: 11
--------------------------------------------------
Epoch    0, Cost: 0.693124
Epoch  500, Cost: 0.372764
Epoch 1000, Cost: 0.353257
Epoch 1500, Cost: 0.343893
Epoch 2000, Cost: 0.339513
Epoch 2500, Cost: 0.334754
Epoch 3000, Cost: 0.328572
Epoch 3500, Cost: 0.325356
Epoch 4000, Cost: 0.319927
Epoch 4500, Cost: 0.316704
--------------------------------------------------
训练完成！


In [8]:
def predict(X_test, parameters, threshold=0.5):
    A2, _ = forward_propagation(X_test, parameters)
    return (A2 > threshold).astype(int)

X_test_T = X_test.T
preds = predict(X_test_T, parameters)

output = pd.DataFrame({
    'PassengerId': test_data.PassengerId,
    'Survived': preds.flatten()
})
output.to_csv('submission.csv', index=False)
print("提交文件已生成：submission.csv")
print(output.head())

提交文件已生成：submission.csv
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
